# Complete Data Processing Pipeline Recreation

This notebook recreates the complete data processing pipeline and final experiments from the `feature_engineering_2.ipynb` project.

## Overview
The pipeline processes raw questionnaire data through multiple stages:
1. Raw data loading and initial processing
2. Missing value handling
3. Questionnaire scoring and totals
4. AQ scoring correction (critical early step)
5. Feature engineering
6. Data standardization and encoding
7. Data balancing
8. Final dataset filtering
9. Experimental setups

## Key Experiments
- Baseline models predicting autism diagnosis
- AQ-based target experiment
- PCA experiment with/without AQ items
- Threshold optimization

## Expected Results
Final PCA + behavioral measures experiment should achieve F1-score of 0.827 and AUC of 0.872 when predicting AQ-based target variable.


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score, 
    f1_score, precision_score, recall_score, precision_recall_curve
)
from sklearn.utils import resample

# Advanced ML libraries
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Set random seeds for reproducibility
np.random.seed(42)
import random
random.seed(42)

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 1. DATA PROCESSING PIPELINE

### A. Raw Data Loading and Initial Processing


In [ ]:
print("="*80)
print("STEP A: RAW DATA LOADING AND INITIAL PROCESSING")
print("="*80)

# Load raw data
raw_data_path = '/Users/eb2007/documents/phd/data/data_c4_raw.csv'
print(f"Loading raw data from: {raw_data_path}")

df = pd.read_csv(raw_data_path)
print(f"Original dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Remove test user IDs (as specified in preprocessing)
print("\nRemoving test user IDs (userid > 174283)...")
df = df[df['userid'] > 174283]
print(f"After removing test users: {df.shape}")

# Create autism_target using diagnosis logic
print("\nCreating autism_target column...")

# Get diagnosis columns
autism_cols = [col for col in df.columns if 'autism_diagnosis' in col]
diagnosis_cols = [col for col in df.columns if col.startswith('diagnosis_') and not 'autism' in col]

print(f"Autism diagnosis columns: {autism_cols}")
print(f"General diagnosis columns: {diagnosis_cols}")

# Method 1: autism_diagnosis_* columns with values ≥1
autism_from_specific = df[autism_cols].fillna(0).ge(1).any(axis=1)

# Method 2: diagnosis_* columns with value = 2
autism_from_general = df[diagnosis_cols].fillna(0).eq(2).any(axis=1)

# Combined target
df['autism_target'] = (autism_from_specific | autism_from_general).astype(int)

print(f"\nAutism target distribution:")
print(df['autism_target'].value_counts())
print(f"Autism percentage: {df['autism_target'].mean()*100:.2f}%")

print(f"\nStep A complete. Dataset shape: {df.shape}")


### B. Missing Value Handling


In [ ]:
print("="*80)
print("STEP B: MISSING VALUE HANDLING")
print("="*80)

# Check missing values before imputation
print("Missing values before imputation:")
missing_before = df.isnull().sum().sort_values(ascending=False)
print(missing_before.head(10))

# Impute demographic columns with 'unknown'
demographic_cols = ['sex', 'handedness', 'education', 'occupation', 'country_region']
print(f"\nImputing demographic columns with 'unknown': {demographic_cols}")

for col in demographic_cols:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')
        print(f"  {col}: {df[col].isnull().sum()} missing values remaining")

# Impute questionnaire scores with median
questionnaire_cols = [col for col in df.columns if any(q in col for q in ['spq_', 'eq_', 'sqr_', 'aq_'])]
print(f"\nImputing questionnaire columns with median: {len(questionnaire_cols)} columns")

df[questionnaire_cols] = df[questionnaire_cols].fillna(df[questionnaire_cols].median())

# Drop rows with missing questionnaire data
print("\nDropping rows with missing questionnaire data...")
df = df.dropna(subset=questionnaire_cols)

print(f"After dropping missing questionnaire data: {df.shape}")

# Check remaining missing values
print("\nMissing values after imputation:")
missing_after = df.isnull().sum().sort_values(ascending=False)
print(missing_after[missing_after > 0].head(10))

print(f"\nStep B complete. Dataset shape: {df.shape}")


### C. Questionnaire Scoring and Totals


In [ ]:
print("="*80)
print("STEP C: QUESTIONNAIRE SCORING AND TOTALS")
print("="*80)

print("Creating questionnaire totals with CORRECT scoring rules...")

# Check what questionnaire columns we have
spq_cols = [col for col in df.columns if col.startswith('spq_')]
eq_cols = [col for col in df.columns if col.startswith('eq_')]
sqr_cols = [col for col in df.columns if col.startswith('sqr_')]
aq_cols = [col for col in df.columns if col.startswith('aq_')]

print(f"Found columns: SPQ={len(spq_cols)}, EQ={len(eq_cols)}, SQR={len(sqr_cols)}, AQ={len(aq_cols)}")

# SPQ-10: Continuous 0-3 scoring (range 0-30)
print("\nSPQ-10: Converting 1-4 scale to 0-3 scale...")
spq_scores = np.zeros(len(df))
for i in range(1, 11):
    col_name = f'spq_{i}'
    if col_name in df.columns:
        # Convert 1,2,3,4 to 0,1,2,3
        spq_scores += (df[col_name] - 1)
df['spq_total'] = spq_scores
print(f"SPQ total range: {df['spq_total'].min()} to {df['spq_total'].max()}")

# EQ-10: Binary 0-1 scoring (range 0-10)
print("\nEQ-10: Converting to binary scoring...")
eq_scores = np.zeros(len(df))
for i in range(1, 11):
    col_name = f'eq_{i}'
    if col_name in df.columns:
        # Binary scoring: responses 1,2 = 1 point, responses 3,4 = 0 points
        eq_scores += ((df[col_name] == 1) | (df[col_name] == 2)).astype(int)
df['eq_total'] = eq_scores
print(f"EQ total range: {df['eq_total'].min()} to {df['eq_total'].max()}")

# SQR-10: Binary 0-1 scoring (range 0-10) 
print("\nSQR-10: Converting to binary scoring...")
sqr_scores = np.zeros(len(df))
for i in range(1, 11):
    col_name = f'sqr_{i}'
    if col_name in df.columns:
        # Binary scoring: responses 1,2 = 1 point, responses 3,4 = 0 points
        sqr_scores += ((df[col_name] == 1) | (df[col_name] == 2)).astype(int)
df['sqr_total'] = sqr_scores
print(f"SQR total range: {df['sqr_total'].min()} to {df['sqr_total'].max()}")

# AQ-10: Will be corrected in next step, but create initial for comparison
aq_cols_list = [f'aq_{i}' for i in range(1, 11)]
available_aq_cols = [col for col in aq_cols_list if col in df.columns]
if available_aq_cols:
    df['aq_total'] = df[available_aq_cols].sum(axis=1)
    print(f"AQ total range (initial, will be corrected): {df['aq_total'].min()} to {df['aq_total'].max()}")
else:
    df['aq_total'] = 0
    print("No AQ columns found, setting aq_total to 0")

# Create D-score (EQ - SQR)
df['d_score'] = df['eq_total'] - df['sqr_total']
print(f"D-score range: {df['d_score'].min()} to {df['d_score'].max()}")

# Show questionnaire total distributions
print("\nQuestionnaire total distributions:")
questionnaire_totals = ['spq_total', 'eq_total', 'sqr_total', 'aq_total', 'd_score']
for col in questionnaire_totals:
    if col in df.columns:
        print(f"  {col}: mean={df[col].mean():.2f}, std={df[col].std():.2f}")

print(f"\nStep C complete. Dataset shape: {df.shape}")


### D. CRITICAL AQ SCORING CORRECTION (Applied Early)

**Why this correction is critical:**
- The AQ-10 has specific scoring rules that differ from simple summation
- Items 1, 7, 8, 10: "Agree" responses (1,2) indicate autistic traits = 1 point each
- Items 2, 3, 4, 5, 6, 9: "Disagree" responses (3,4) indicate autistic traits = 1 point each
- This correction ensures accurate AQ scoring for all subsequent analyses


In [ ]:
print("="*80)
print("STEP D: CRITICAL AQ SCORING CORRECTION")
print("="*80)

print("Applying official AQ-10 scoring rules...")
print("Items 1, 7, 8, 10: Agree (responses 1,2) = 1 point each")
print("Items 2, 3, 4, 5, 6, 9: Disagree (responses 3,4) = 1 point each")

# Check what AQ columns we have
aq_cols = [col for col in df.columns if col.startswith('aq_')]
print(f"Found AQ columns: {aq_cols}")

# AQ-10 official scoring rules
agree_items = [1, 7, 8, 10]  # Items where "agree" indicates autistic trait
disagree_items = [2, 3, 4, 5, 6, 9]  # Items where "disagree" indicates autistic trait

aq_scores = np.zeros(len(df))

# Score agree items (responses 1,2 = agree = 1 point)
for item_num in agree_items:
    col_name = f'aq_{item_num}'
    if col_name in df.columns:
        aq_scores += ((df[col_name] == 1) | (df[col_name] == 2)).astype(int)
        print(f"Scored {col_name}: {((df[col_name] == 1) | (df[col_name] == 2)).sum()} agree responses")
    else:
        print(f"Missing column: {col_name}")

# Score disagree items (responses 3,4 = disagree = 1 point)
for item_num in disagree_items:
    col_name = f'aq_{item_num}'
    if col_name in df.columns:
        aq_scores += ((df[col_name] == 3) | (df[col_name] == 4)).astype(int)
        print(f"Scored {col_name}: {((df[col_name] == 3) | (df[col_name] == 4)).sum()} disagree responses")
    else:
        print(f"Missing column: {col_name}")

# Update AQ total with correct scoring
df['aq_total'] = aq_scores

print(f"\nAQ total range (corrected): {df['aq_total'].min()} to {df['aq_total'].max()}")
print(f"Cases with AQ >= 6: {len(df[df['aq_total'] >= 6])}")
print(f"Cases with AQ < 6: {len(df[df['aq_total'] < 6])}")

# Show AQ distribution by autism status
print("\nAQ distribution by autism status:")
aq_by_autism = df.groupby('autism_target')['aq_total'].agg(['count', 'mean', 'std'])
print(aq_by_autism)

# Verify AQ scoring is correct
print(f"\n AQ-10 scoring verification:")
print(f"   Score range: {df['aq_total'].min()}-{df['aq_total'].max()} (should be 0-10)")
print(f"   Clinical threshold (AQ≥6): {len(df[df['aq_total'] >= 6])} cases")
print(f"   Autism cases with AQ≥6: {len(df[(df['autism_target']==1) & (df['aq_total']>=6)])}")

print(f"\nStep D complete. Dataset shape: {df.shape}")


### E. Feature Engineering (Basic - Matching feature_engineering.ipynb)


In [ ]:
# CORRECTED STEP E: FEATURE ENGINEERING - FIXED SEX MAPPING
print("="*80)
print("CORRECTED STEP E: FEATURE ENGINEERING")
print("="*80)

print("Creating engineered features matching feature_engineering.ipynb...")

# Age group bins
print("Creating age group bins...")
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 30, 45, 60, 100], 
                        labels=['0-18', '19-30', '31-45', '46-60', '61+'])
print(f"Age groups created: {df['age_group'].value_counts().to_dict()}")

# Non-linear transformations
print("\nCreating non-linear transformations...")
df['log_aq_total'] = np.log1p(df['aq_total'])
df['sqrt_age'] = np.sqrt(df['age'])
print("Created log_aq_total and sqrt_age")

# Interaction terms
print("\nCreating interaction terms...")
df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
df['sqp_aq_interaction'] = df['spq_total'] * df['aq_total']
df['age_x_eq'] = df['age'] * df['eq_total']
print("Created aq_eq_interaction, sqp_aq_interaction, age_x_eq")

# Questionnaire score ratios
print("\nCreating questionnaire ratios...")
df['aq_spq_ratio'] = df['aq_total'] / (df['spq_total'] + 1e-8)
df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
print("Created aq_spq_ratio and eq_sqr_ratio")

# High AQ threshold (using corrected AQ scoring)
print("\nCreating high AQ threshold...")
df['high_aq'] = (df['aq_total'] >= 6).astype(int)
print(f"High AQ cases (>=6): {df['high_aq'].sum()}")

# STEM occupation detection
print("\nCreating STEM occupation feature...")
stem_occupation_codes = {2, 3, 5, 21}
def is_stem(occupation_code):
    try:
        return int(float(occupation_code) in stem_occupation_codes)
    except:
        return 0

if 'occupation' in df.columns:
    df['is_stem_occupation'] = df['occupation'].apply(is_stem)
    print(f"STEM occupation cases: {df['is_stem_occupation'].sum()}")
else:
    df['is_stem_occupation'] = 0
    print("Occupation column not found, created dummy is_stem_occupation")

# CORRECTED Sex mapping to numeric
print("\nCreating sex numeric mapping...")
if 'sex' in df.columns:
    # Original sex column exists - map it directly
    sex_map = {1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3}  # male=0, female=1, other=2, prefer_not_to_say=3
    df['sex_num'] = df['sex'].map(sex_map)
    print(f"Sex distribution: {df['sex_num'].value_counts().to_dict()}")
    print("Sex mapping successful - no zero variance issue")
else:
    # If sex column doesn't exist, this indicates an error in missing value handling
    print("ERROR: Sex column missing - check missing value handling")
    print("Creating fallback sex_num column...")
    df['sex_num'] = 0  # Default fallback
    print("Fallback sex mapping created")

# Additional interactions (CORRECTED)
print("\nCreating additional interactions...")
df['age_x_aq'] = df['age'] * df['aq_total']
df['sex_x_eq'] = df['sex_num'] * df['eq_total']

# Handle columns that may have been dropped during standardization
if 'handedness' in df.columns:
    df['handedness_x_aq'] = df['handedness'].replace('unknown', 0).astype(float) * df['aq_total']
    print("Created handedness_x_aq")
else:
    df['handedness_x_aq'] = 0  # Create dummy column
    print("handedness column not found, created dummy handedness_x_aq")

if 'education' in df.columns:
    df['education_x_aq'] = df['education'].replace('unknown', 0).astype(float) * df['aq_total']
    print("Created education_x_aq")
else:
    df['education_x_aq'] = 0  # Create dummy column
    print("education column not found, created dummy education_x_aq")

print("Created age_x_aq, sex_x_eq, handedness_x_aq, education_x_aq")

print(f"\nCorrected Step E complete. Dataset shape: {df.shape}")
print(f"Total features created: {len(df.columns)}")

### F. Data Standardization and Encoding


In [ ]:
# CORRECTED STEP F: DATA STANDARDIZATION AND ENCODING
print("="*80)
print("CORRECTED STEP F: DATA STANDARDIZATION AND ENCODING")
print("="*80)

# Apply StandardScaler to questionnaire items (but preserve raw totals for filtering)
print("Standardizing questionnaire items...")
questionnaire_cols = [col for col in df.columns if col.startswith(('spq_', 'eq_', 'sqr_', 'aq_'))]
# Exclude totals from standardization - we need raw scores for filtering
individual_item_cols = [col for col in questionnaire_cols if not col.endswith('_total')]
scaler = StandardScaler()
if individual_item_cols:
    df[individual_item_cols] = scaler.fit_transform(df[individual_item_cols])
    print(f"Standardized {len(individual_item_cols)} individual questionnaire items")
    print("Preserved raw totals (spq_total, eq_total, sqr_total, aq_total) for filtering")

# One-hot encode age groups (sex will be handled separately)
print("\nOne-hot encoding age groups...")
df = pd.get_dummies(df, columns=['age_group'], drop_first=True)
print("Age groups one-hot encoded")

# Remove data leakage columns (all diagnosis-related columns)
print("\nRemoving data leakage columns...")
diagnosis_cols = [col for col in df.columns if col.startswith('diagnosis_') or col.startswith('autism_diagnosis')]
df = df.drop(columns=diagnosis_cols, errors='ignore')
print(f"Removed {len(diagnosis_cols)} diagnosis columns")

# Drop unnecessary columns
print("\nDropping unnecessary columns...")
drop_cols = ['userid', 'repeat', 'occupation', 'country_region', 'handedness', 'education']
drop_cols += [col for col in df.columns if col.startswith('occupation_') or col.startswith('country_region_') or col.startswith('handedness_') or col.startswith('education_')]
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
print(f"Dropped unnecessary columns")

# Fill remaining NaNs with 0
print("\nFilling remaining NaNs with 0...")
df = df.fillna(0)

# Save processed data
print("\nSaving processed data...")
output_path = 'data/processed/data_c4_processed.csv'
import os
os.makedirs('data/processed', exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Processed data saved to {output_path}. Shape: {df.shape}")

print(f"\nStep F complete. Dataset shape: {df.shape}")
print(f"Final feature count: {len(df.columns)}")

In [ ]:
# CORRECTED SEX ONE-HOT ENCODING - FIXING DATA LEAKAGE
print("="*80)
print("CORRECTED SEX ONE-HOT ENCODING - FIXING DATA LEAKAGE")
print("="*80)

# Check what sex columns currently exist
print("\nChecking current sex columns...")
sex_cols = [col for col in df.columns if col.startswith('sex_')]
print(f"Current sex columns: {sex_cols}")

# CRITICAL FIX: Remove problematic interaction columns
problematic_cols = ['sex_x_eq', 'sex_unknown']
cols_to_remove = [col for col in problematic_cols if col in df.columns]

if cols_to_remove:
    print(f"\nRemoving problematic sex columns: {cols_to_remove}")
    df = df.drop(columns=cols_to_remove, errors='ignore')
    print(f"Removed {len(cols_to_remove)} problematic columns")

# Re-check sex columns after cleanup
sex_cols = [col for col in df.columns if col.startswith('sex_')]
print(f"Cleaned sex columns: {sex_cols}")

# Verify the remaining encoding
if len(sex_cols) > 0:
    print(f"\nVerifying remaining sex encoding...")
    total_sex_sum = sum(df[col].sum() for col in sex_cols)
    expected_sum = len(df)
    print(f"Sex columns sum: {total_sex_sum:,}")
    print(f"Expected sum: {expected_sum:,}")
    
    if abs(total_sex_sum - expected_sum) < expected_sum * 0.05:  # Within 5%
        print("✓ Sex encoding looks correct")
    else:
        print("⚠ Sex encoding still has issues")
        
        # Show individual column sums for debugging
        print("\nIndividual sex column sums:")
        for col in sex_cols:
            col_sum = df[col].sum()
            print(f"  {col}: {col_sum:,}")
else:
    print("ERROR: No sex columns remaining after cleanup")

print(f"\nCorrected sex one-hot encoding complete. Dataset shape: {df.shape}")

### G. Data Balancing (experiments/experiment_matched_sample.py)


In [ ]:
print("="*80)
print("STEP G: DATA BALANCING")
print("="*80)

# Load processed data
print("Loading processed data...")
df_processed = pd.read_csv('data/processed/data_c4_processed.csv')
print(f"Processed dataset shape: {df_processed.shape}")

# Separate autistic and non-autistic cases
print("\nSeparating autistic and non-autistic cases...")
autistic_df = df_processed[df_processed['autism_target'] == 1]
non_autistic_df = df_processed[df_processed['autism_target'] == 0]

print(f"Autistic cases: {len(autistic_df)}")
print(f"Non-autistic cases: {len(non_autistic_df)}")
print(f"Class imbalance ratio: {len(non_autistic_df)/len(autistic_df):.2f}:1")

# Downsample non-autistic to match autistic count
print("\nDownsampling non-autistic cases to match autistic count...")
non_autistic_down = resample(non_autistic_df, replace=False, n_samples=len(autistic_df), random_state=42)

# Combine balanced datasets
df_balanced = pd.concat([autistic_df, non_autistic_down])

# Shuffle combined dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print(f"Balanced autism target distribution:")
print(df_balanced['autism_target'].value_counts())

# Save balanced dataset
balanced_path = 'data/processed/data_c4_matched_balanced.csv'
df_balanced.to_csv(balanced_path, index=False)
print(f"\nBalanced dataset saved to {balanced_path}")

print(f"\nStep G complete. Dataset shape: {df_balanced.shape}")


### H. Final Dataset Filtering

**Critical Filtering Step:** Remove individuals who have `autism_target = 1` (diagnosed with autism) but have `AQ score < 6`. This creates the "corrected" dataset used for final experiments.

**Rationale:** This filtering removes diagnosed individuals who don't meet the AQ threshold, creating a cleaner autism diagnosis dataset for the experiments.


In [ ]:
print("="*80)
print("STEP H: FINAL DATASET FILTERING")
print("="*80)

# Load balanced dataset
print("Loading balanced dataset...")
df_balanced = pd.read_csv('data/processed/data_c4_matched_balanced.csv')
print(f"Balanced dataset shape: {df_balanced.shape}")

# Show distribution before filtering
print("\nDistribution before filtering:")
print(f"Total cases: {len(df_balanced)}")
print(f"Autism cases: {len(df_balanced[df_balanced['autism_target'] == 1])}")
print(f"Non-autism cases: {len(df_balanced[df_balanced['autism_target'] == 0])}")

# Check AQ distribution by autism status (these are RAW scores, not standardized)
print("\nAQ distribution by autism status (RAW scores):")
aq_by_autism = df_balanced.groupby('autism_target')['aq_total'].agg(['count', 'mean', 'std', 'min', 'max'])
print(aq_by_autism)

# Count cases to be removed
autism_cases = df_balanced[df_balanced['autism_target'] == 1]
low_aq_autism = autism_cases[autism_cases['aq_total'] < 6]
print(f"\nAutism cases with AQ < 6: {len(low_aq_autism)}")
print(f"Percentage of autism cases with AQ < 6: {len(low_aq_autism)/len(autism_cases)*100:.1f}%")

# Apply filtering: Remove diagnosed individuals with AQ < 6
print("\nApplying filtering: Remove autism cases with AQ < 6...")
df_filtered = df_balanced[~((df_balanced['autism_target'] == 1) & (df_balanced['aq_total'] < 6))]

print(f"\nDistribution after filtering:")
print(f"Total cases: {len(df_filtered)}")
print(f"Autism cases: {len(df_filtered[df_filtered['autism_target'] == 1])}")
print(f"Non-autism cases: {len(df_filtered[df_filtered['autism_target'] == 0])}")
print(f"Cases removed: {len(df_balanced) - len(df_filtered)}")

# Check AQ distribution after filtering
print("\nAQ distribution after filtering:")
aq_by_autism_filtered = df_filtered.groupby('autism_target')['aq_total'].agg(['count', 'mean', 'std', 'min', 'max'])
print(aq_by_autism_filtered)

# Verify we have both classes
if len(df_filtered[df_filtered['autism_target'] == 1]) == 0:
    print("\n  WARNING: No autism cases remaining after filtering!")
    print("This suggests the AQ threshold of 6 may be too high.")
    print("Consider using a lower threshold or checking the AQ scoring.")
elif len(df_filtered[df_filtered['autism_target'] == 0]) == 0:
    print("\n  WARNING: No non-autism cases remaining after filtering!")
else:
    print(f"\n SUCCESS: Both classes present after filtering")
    print(f"   Autism cases: {len(df_filtered[df_filtered['autism_target'] == 1])}")
    print(f"   Non-autism cases: {len(df_filtered[df_filtered['autism_target'] == 0])}")

# Save final filtered dataset
final_path = 'data/processed/data_c4_final_recreated.csv'
df_filtered.to_csv(final_path, index=False)
print(f"\nFinal filtered dataset saved to {final_path}")

print(f"\nStep H complete. Final dataset shape: {df_filtered.shape}")
print(f"Final feature count: {len(df_filtered.columns)}")


In [ ]:
print("="*80)
print("STEP I: POST-FILTERING REBALANCING")
print("="*80)

# Load filtered dataset
print("Loading filtered dataset...")
df_filtered = pd.read_csv('data/processed/data_c4_final_recreated.csv')
print(f"Filtered dataset shape: {df_filtered.shape}")

# Show current distribution
print("\nCurrent distribution after filtering:")
autism_cases = len(df_filtered[df_filtered['autism_target'] == 1])
non_autism_cases = len(df_filtered[df_filtered['autism_target'] == 0])
print(f"Autism cases: {autism_cases}")
print(f"Non-autism cases: {non_autism_cases}")
print(f"Current ratio: {non_autism_cases/autism_cases:.2f}:1")

# Rebalance to 50/50 (match original notebook)
print("\nRebalancing to 50/50 distribution...")
autistic_df = df_filtered[df_filtered['autism_target'] == 1]
non_autistic_df = df_filtered[df_filtered['autism_target'] == 0]

# Downsample non-autistic to match autistic count
non_autistic_rebalanced = resample(non_autistic_df, replace=False, n_samples=len(autistic_df), random_state=42)

# Combine rebalanced datasets
df_final_balanced = pd.concat([autistic_df, non_autistic_rebalanced])

# Shuffle final dataset
df_final_balanced = df_final_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal balanced dataset shape: {df_final_balanced.shape}")
print(f"Final autism target distribution:")
print(df_final_balanced['autism_target'].value_counts())

# Save final balanced dataset
final_balanced_path = 'data/processed/data_c4_final_recreated.csv'
df_final_balanced.to_csv(final_balanced_path, index=False)
print(f"\nFinal balanced dataset saved to {final_balanced_path}")

print(f"\nStep I complete. Final dataset shape: {df_final_balanced.shape}")
print(f"Final feature count: {len(df_final_balanced.columns)}")

# Verify we match original notebook size
expected_size = 20973
actual_size = len(df_final_balanced)
print(f"\n Size verification:")
print(f"   Expected size (original): {expected_size}")
print(f"   Actual size (recreated): {actual_size}")
print(f"   Match: {' YES' if abs(actual_size - expected_size) < 1000 else ' NO'}")


In [ ]:
# CORRECTED DATA CLEANING - FIXING BALANCE AND SEX ENCODING ISSUES
print("="*80)
print("CORRECTED DATA CLEANING - FIXING IDENTIFIED ISSUES")
print("="*80)

# Load final dataset
df_final = pd.read_csv('data/processed/data_c4_final_recreated.csv')
print(f"Original dataset shape: {df_final.shape}")

# 1. FIXING DUPLICATE ROWS WITH BALANCE PRESERVATION
print("\n" + "-"*40)
print("1. FIXING DUPLICATE ROWS WITH BALANCE PRESERVATION")
print("-"*40)

# Check balance before duplicate removal
target_dist_before = df_final['autism_target'].value_counts()
print(f"Balance before: Autism={target_dist_before[1]}, Non-autism={target_dist_before[0]}")

# Remove duplicates while preserving balance
df_cleaned = df_final.drop_duplicates()
duplicates_removed = len(df_final) - len(df_cleaned)
print(f"Removed {duplicates_removed} duplicate rows")

# Check balance after duplicate removal
target_dist_after = df_cleaned['autism_target'].value_counts()
print(f"Balance after: Autism={target_dist_after[1]}, Non-autism={target_dist_after[0]}")

# Rebalance if needed (CRITICAL FIX)
if target_dist_after[0] != target_dist_after[1]:
    print("Rebalancing dataset after duplicate removal...")
    autistic_df = df_cleaned[df_cleaned['autism_target'] == 1]
    non_autistic_df = df_cleaned[df_cleaned['autism_target'] == 0]
    
    # Use the smaller class size
    min_size = min(len(autistic_df), len(non_autistic_df))
    
    # Downsample both classes to match the smaller size
    autistic_rebalanced = resample(autistic_df, replace=False, n_samples=min_size, random_state=42)
    non_autistic_rebalanced = resample(non_autistic_df, replace=False, n_samples=min_size, random_state=42)
    
    # Combine and shuffle
    df_cleaned = pd.concat([autistic_rebalanced, non_autistic_rebalanced])
    df_cleaned = df_cleaned.sample(frac=1, random_state=42).reset_index(drop=True)
    
    target_dist_final = df_cleaned['autism_target'].value_counts()
    print(f"Final balance: Autism={target_dist_final[1]}, Non-autism={target_dist_final[0]}")

print(f"Dataset shape: {df_final.shape} → {df_cleaned.shape}")

# 2. FIXING ZERO VARIANCE FEATURES
print("\n" + "-"*40)
print("2. FIXING ZERO VARIANCE FEATURES")
print("-"*40)

numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
zero_var_features = []
for col in numeric_cols:
    if col != 'autism_target' and df_cleaned[col].var() == 0:
        zero_var_features.append(col)

if zero_var_features:
    print(f"Removed zero variance features: {zero_var_features}")
    df_cleaned = df_cleaned.drop(columns=zero_var_features, errors='ignore')
    print(f"Features: {len(df_final.columns)} → {len(df_cleaned.columns)}")
else:
    print("No zero variance features found")

# 3. FIXING SEX ENCODING ISSUES (CRITICAL FIX)
print("\n" + "-"*40)
print("3. FIXING SEX ENCODING ISSUES")
print("-"*40)

# Remove problematic sex interaction columns
problematic_sex_cols = ['sex_x_eq']  # These create inflated sums
sex_cols_to_remove = [col for col in df_cleaned.columns if col in problematic_sex_cols]

if sex_cols_to_remove:
    print(f"Removing problematic sex columns: {sex_cols_to_remove}")
    df_cleaned = df_cleaned.drop(columns=sex_cols_to_remove, errors='ignore')
    print(f"Features after removal: {len(df_cleaned.columns)}")

# Verify remaining sex columns
sex_cols = [col for col in df_cleaned.columns if col.startswith('sex_')]
if sex_cols:
    print("Remaining sex columns:")
    total_sex_sum = 0
    for col in sex_cols:
        count = df_cleaned[col].sum()
        total_sex_sum += count
        print(f"  {col}: {count:,} ({count/len(df_cleaned)*100:.1f}%)")
    
    expected_sum = len(df_cleaned)
    print(f"\nTotal sex column sum: {total_sex_sum:,}")
    print(f"Expected sum: {expected_sum:,}")
    
    if abs(total_sex_sum - expected_sum) < expected_sum * 0.1:  # Within 10%
        print("✓ Sex encoding looks correct")
    else:
        print("⚠ Sex encoding still has issues")
else:
    print("No sex columns found")

# 4. SAVING CLEANED DATASET
print("\n" + "-"*40)
print("4. SAVING CLEANED DATASET")
print("-"*40)

cleaned_path = 'data/processed/data_c4_final_recreated_cleaned.csv'
df_cleaned.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved to {cleaned_path}")

print("\n" + "="*60)
print("FINAL CLEANED DATASET VERIFICATION")
print("="*60)

print(f"Final dataset shape: {df_cleaned.shape}")
print(f"Missing values: {df_cleaned.isnull().sum().sum()}")
print(f"Duplicate rows: {df_cleaned.duplicated().sum()}")

# Check for zero variance features again
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
zero_var_features = []
for col in numeric_cols:
    if col != 'autism_target' and df_cleaned[col].var() == 0:
        zero_var_features.append(col)

print(f"Zero variance features: {len(zero_var_features)}")
if len(zero_var_features) == 0:
    print("   No zero variance features")
else:
    print(f"   Zero variance features: {zero_var_features}")

# Updated modeling readiness checklist
target_dist = df_cleaned['autism_target'].value_counts()
readiness_checks = {
    "Balanced dataset": target_dist[0] == target_dist[1],
    "No missing values": df_cleaned.isnull().sum().sum() == 0,
    "No duplicates": df_cleaned.duplicated().sum() == 0,
    "All autism cases AQ ≥ 6": len(df_cleaned[(df_cleaned['autism_target'] == 1) & (df_cleaned['aq_total'] >= 6)]) == len(df_cleaned[df_cleaned['autism_target'] == 1]),
    "Reasonable sample size": len(df_cleaned) > 10000,
    "Sufficient features": len(df_cleaned.columns) > 50,
    "No zero variance features": len(zero_var_features) == 0
}

print("\nUpdated modeling readiness checklist:")
all_ready = True
for check, passed in readiness_checks.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {check}: {status}")
    if not passed:
        all_ready = False

print(f"\nFinal readiness: {'✓ READY FOR MODELING' if all_ready else '✗ ISSUES REMAIN'}")

print("\n" + "="*80)
print("CORRECTED DATA CLEANING COMPLETE")
print("="*80)


In [ ]:
print("="*80)
print("COMPREHENSIVE DATASET ANALYSIS - PRE-MODELING")
print("="*80)

# Load cleaned final dataset (after duplicate removal and rebalancing)
df_final = pd.read_csv('data/processed/data_c4_final_recreated_cleaned.csv')
print(f"Final dataset shape: {df_final.shape}")
print(f"Features: {df_final.shape[1] - 1} (excluding target)")
print(f"Samples: {df_final.shape[0]}")

# 1. TARGET VARIABLE ANALYSIS
print("\n" + "="*60)
print("1. TARGET VARIABLE ANALYSIS")
print("="*60)

target_dist = df_final['autism_target'].value_counts()
print(f"Target distribution:")
print(f"  Autism cases (1): {target_dist[1]:,} ({target_dist[1]/len(df_final)*100:.1f}%)")
print(f"  Non-autism cases (0): {target_dist[0]:,} ({target_dist[0]/len(df_final)*100:.1f}%)")
print(f"  Balance ratio: {target_dist[0]/target_dist[1]:.3f}:1")

# 2. QUESTIONNAIRE SCORES ANALYSIS
print("\n" + "="*60)
print("2. QUESTIONNAIRE SCORES ANALYSIS")
print("="*60)

questionnaire_totals = ['spq_total', 'eq_total', 'sqr_total', 'aq_total', 'd_score']
print("Questionnaire score ranges and means:")
for col in questionnaire_totals:
    if col in df_final.columns:
        print(f"  {col}:")
        print(f"    Range: {df_final[col].min():.1f} to {df_final[col].max():.1f}")
        print(f"    Mean: {df_final[col].mean():.2f} ± {df_final[col].std():.2f}")

# AQ distribution by target
print(f"\nAQ score distribution by autism status:")
aq_by_target = df_final.groupby('autism_target')['aq_total'].agg(['count', 'mean', 'std', 'min', 'max'])
print(aq_by_target)

# Verify AQ threshold compliance
autism_aq6_plus = len(df_final[(df_final['autism_target'] == 1) & (df_final['aq_total'] >= 6)])
autism_total = len(df_final[df_final['autism_target'] == 1])
print(f"\nAQ threshold verification:")
print(f"  Autism cases with AQ ≥ 6: {autism_aq6_plus}/{autism_total} ({autism_aq6_plus/autism_total*100:.1f}%)")
print(f"   All autism cases have AQ ≥ 6: {'YES' if autism_aq6_plus == autism_total else 'NO'}")

# 3. DEMOGRAPHIC ANALYSIS
print("\n" + "="*60)
print("3. DEMOGRAPHIC ANALYSIS")
print("="*60)

# Age analysis
if 'age' in df_final.columns:
    print(f"Age distribution:")
    print(f"  Range: {df_final['age'].min():.0f} to {df_final['age'].max():.0f} years")
    print(f"  Mean: {df_final['age'].mean():.1f} ± {df_final['age'].std():.1f} years")
    
    # Age by target
    age_by_target = df_final.groupby('autism_target')['age'].agg(['mean', 'std'])
    print(f"  Mean age by autism status:")
    print(f"    Autism: {age_by_target.loc[1, 'mean']:.1f} ± {age_by_target.loc[1, 'std']:.1f}")
    print(f"    Non-autism: {age_by_target.loc[0, 'mean']:.1f} ± {age_by_target.loc[0, 'std']:.1f}")

# Sex analysis - CORRECTED FOR PROPER SEX ENCODING
sex_cols = [col for col in df_final.columns if col.startswith('sex_')]
if sex_cols:
    print(f"\nSex distribution:")
    
    # Check if we have proper one-hot encoding
    one_hot_cols = [col for col in sex_cols if col in ['sex_2.0', 'sex_3.0', 'sex_4.0']]
    numeric_col = 'sex_num' if 'sex_num' in sex_cols else None
    
    if one_hot_cols:
        print("  One-hot encoded sex categories:")
        total_one_hot = 0
        for col in one_hot_cols:
            count = df_final[col].sum()
            total_one_hot += count
            category_name = {'sex_2.0': 'Female', 'sex_3.0': 'Transgender/Other', 'sex_4.0': 'Prefer not to say'}[col]
            print(f"    {category_name} ({col}): {count:,} ({count/len(df_final)*100:.1f}%)")
        
        # Calculate implicit Male count (reference category)
        male_count = len(df_final) - total_one_hot
        print(f"    Male (implicit): {male_count:,} ({male_count/len(df_final)*100:.1f}%)")
        
        # Verify encoding
        expected_total = len(df_final)
        actual_total = total_one_hot + male_count
        if abs(actual_total - expected_total) < expected_total * 0.01:  # Within 1%
            print("  ✓ Sex encoding verified: All categories sum to sample count")
        else:
            print(f"  ⚠ Sex encoding issue: Expected {expected_total}, got {actual_total}")
    
    if numeric_col:
        print(f"\n  Numeric sex mapping ({numeric_col}):")
        sex_counts = df_final[numeric_col].value_counts().sort_index()
        for val, count in sex_counts.items():
            category_name = {0: 'Male', 1: 'Female', 2: 'Transgender/Other', 3: 'Prefer not to say'}[val]
            print(f"    {category_name} ({val}): {count:,} ({count/len(df_final)*100:.1f}%)")
    
    # Check for problematic columns
    problematic_cols = [col for col in sex_cols if col not in one_hot_cols + [numeric_col]]
    if problematic_cols:
        print(f"\n  ⚠ Problematic sex columns found: {problematic_cols}")
        print("    These should be removed to avoid data leakage")
    
else:
    print(f"\nSex distribution: No sex columns found")

# 4. FEATURE ENGINEERING VERIFICATION
print("\n" + "="*60)
print("4. FEATURE ENGINEERING VERIFICATION")
print("="*60)

# Check engineered features
engineered_features = ['log_aq_total', 'sqrt_age', 'aq_eq_interaction', 'sqp_aq_interaction', 
                       'age_x_eq', 'aq_spq_ratio', 'eq_sqr_ratio', 'high_aq', 'is_stem_occupation']

print("Engineered features present:")
for feat in engineered_features:
    if feat in df_final.columns:
        print(f"   {feat}")
    else:
        print(f"   {feat} - MISSING")

# Check interaction terms
interaction_features = [col for col in df_final.columns if '_x_' in col or '_interaction' in col]
print(f"\nInteraction features found: {len(interaction_features)}")
for feat in interaction_features[:5]:  # Show first 5
    print(f"  {feat}")

# 5. DATA QUALITY CHECKS
print("\n" + "="*60)
print("5. DATA QUALITY CHECKS")
print("="*60)

# Missing values
missing_count = df_final.isnull().sum().sum()
print(f"Missing values: {missing_count}")
if missing_count == 0:
    print("   No missing values")
else:
    print("   Missing values present")

# Duplicate rows
duplicate_count = df_final.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")
if duplicate_count == 0:
    print("   No duplicate rows")
else:
    print("   Duplicate rows present")

# Zero variance features
numeric_cols = df_final.select_dtypes(include=[np.number]).columns
zero_var_features = []
for col in numeric_cols:
    if col != 'autism_target' and df_final[col].var() == 0:
        zero_var_features.append(col)

print(f"Zero variance features: {len(zero_var_features)}")
if len(zero_var_features) == 0:
    print("   No zero variance features")
else:
    print(f"   Zero variance features: {zero_var_features}")

# 6. COMPARISON WITH ORIGINAL NOTEBOOK
print("\n" + "="*60)
print("6. COMPARISON WITH ORIGINAL NOTEBOOK")
print("="*60)

original_size = 20973
actual_size = len(df_final)
size_diff = actual_size - original_size
size_diff_pct = (size_diff / original_size) * 100

print(f"Dataset size comparison:")
print(f"  Original notebook: {original_size:,} samples")
print(f"  Recreated notebook: {actual_size:,} samples")
print(f"  Difference: {size_diff:+,} samples ({size_diff_pct:+.1f}%)")

if abs(size_diff_pct) < 10:
    print("   Size matches original closely")
elif size_diff_pct > 0:
    print("  ℹ Larger dataset (due to raw AQ scoring vs transformed)")
else:
    print("   Smaller dataset than expected")

# 7. READINESS FOR MODELING
print("\n" + "="*60)
print("7. READINESS FOR MODELING")
print("="*60)

readiness_checks = {
    "Balanced dataset": target_dist[0] == target_dist[1],
    "No missing values": missing_count == 0,
    "No duplicates": duplicate_count == 0,
    "All autism cases AQ ≥ 6": autism_aq6_plus == autism_total,
    "Reasonable sample size": actual_size > 10000,
    "Sufficient features": df_final.shape[1] > 50
}

print("Modeling readiness checklist:")
all_ready = True
for check, passed in readiness_checks.items():
    status = " PASS" if passed else " FAIL"
    print(f"  {check}: {status}")
    if not passed:
        all_ready = False

print(f"\nOverall readiness: {' READY FOR MODELING' if all_ready else ' ISSUES TO ADDRESS'}")

print(f"\n" + "="*80)
print("DATASET ANALYSIS COMPLETE")
print("="*80)


## 2. EXPERIMENTAL SETUPS

### A. Baseline Models: Predicting Autism Target (Without AQ Items)

**Objective**: Train baseline models to predict autism diagnosis using behavioral measures (SPQ, EQ, SQ-R) and demographics, **excluding AQ items** to avoid circularity.

**Rationale**: This establishes baseline performance using non-AQ measures, which is important for understanding the contribution of AQ items in subsequent experiments.

**Expected Performance**: Lower than AQ-inclusive models, but establishes meaningful baseline for comparison.


### CORRECTED EXPERIMENT A: Baseline Models (Fixed Data Type Issue)

**Issue Fixed**: The original experiment failed because categorical columns with 'unknown' values couldn't be converted to float for StandardScaler. This corrected version properly handles categorical data conversion.


### DATA LEAKAGE INVESTIGATION RESULTS

**CRITICAL ISSUE FOUND**: The `high_aq` feature creates perfect data leakage!

**Problem**: 
- `high_aq = (aq_total >= 6)` 
- All autism cases in final dataset have AQ ≥ 6 (due to filtering)
- Therefore: 100% of autism cases have `high_aq = 1`
- This creates a perfect predictor with 0.83 correlation

**Solution**: Exclude `high_aq` from baseline experiments to avoid circularity.


In [ ]:
# CORRECTED EXPERIMENT A: Baseline Models (Complete AQ Exclusion)
print("="*80)
print("CORRECTED EXPERIMENT A: BASELINE MODELS - COMPLETE AQ EXCLUSION")
print("="*80)

# Load final cleaned dataset
df_final = pd.read_csv('data/processed/data_c4_final_recreated_cleaned.csv')
print(f"Dataset shape: {df_final.shape}")

# Prepare features and target
print("\nPreparing features and target...")

# Remove ALL AQ-related features (including interactions)
aq_features = [col for col in df_final.columns if 'aq_' in col.lower()]
aq_interaction_features = [col for col in df_final.columns if 'aq' in col.lower() and col not in aq_features]
leakage_features = ['high_aq']
features_to_exclude = ['autism_target'] + aq_features + aq_interaction_features + leakage_features

print(f"AQ item features to exclude: {aq_features}")
print(f"AQ interaction features to exclude: {aq_interaction_features}")
print(f"Data leakage features to exclude: {leakage_features}")

# Create feature set excluding ALL AQ-related features
feature_cols = [col for col in df_final.columns if col not in features_to_exclude]
print(f"Feature columns ({len(feature_cols)}): {feature_cols[:10]}...")

X = df_final[feature_cols]
y = df_final['autism_target']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle data types and missing values
print("\nHandling data types and missing values...")
missing_before = X.isnull().sum().sum()
print(f"Missing values before: {missing_before}")

# Convert categorical columns to numeric codes
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print(f"Categorical columns found: {list(categorical_cols)}")
    for col in categorical_cols:
        print(f"  Converting {col} to numeric codes")
        X[col] = pd.Categorical(X[col]).codes

# Handle missing values
if missing_before > 0:
    X = X.fillna(X.median())

missing_after = X.isnull().sum().sum()
print(f"Missing values after: {missing_after}")

# Check for any remaining high correlations
print(f"\nChecking for data leakage...")
feature_correlations = []
for col in X.columns:
    if X[col].dtype in ['int64', 'float64']:
        corr = X[col].corr(y)
        feature_correlations.append((col, abs(corr), corr))

feature_correlations.sort(key=lambda x: x[1], reverse=True)
print(f"Top 5 feature correlations with target:")
for col, abs_corr, corr in feature_correlations[:5]:
    print(f"  {col}: {corr:.4f}")

high_corr_features = [col for col, abs_corr, corr in feature_correlations if abs_corr > 0.7]
if high_corr_features:
    print(f"\n⚠️ WARNING: High correlation features found: {high_corr_features}")
else:
    print(f"\n✅ No high correlation features (correlation < 0.7)")

# Scale features
print("\nScaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print(f"Scaled features shape: {X_scaled.shape}")
print(f"Feature means: {X_scaled.mean().mean():.6f}")
print(f"Feature stds: {X_scaled.std().mean():.6f}")

# Split data
print("\nSplitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

# Train and evaluate models
print("\n" + "="*60)
print("TRAINING AND EVALUATING BASELINE MODELS (COMPLETE AQ EXCLUSION)")
print("="*60)

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results[name] = {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  AUC: {auc:.4f}")

# Summary of results
print("\n" + "="*60)
print("CORRECTED BASELINE MODELS SUMMARY (COMPLETE AQ EXCLUSION)")
print("="*60)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('auc', ascending=False)

print("\nResults ranked by AUC:")
print(results_df.round(4))

# Save results
results_df.to_csv('data/processed/baseline_models_complete_aq_exclusion_results.csv')
print(f"\nResults saved to: data/processed/baseline_models_complete_aq_exclusion_results.csv")

print("\n" + "="*80)
print("CORRECTED EXPERIMENT A COMPLETE (COMPLETE AQ EXCLUSION)")
print("="*80)

### B. PCA
**Objective**: how can we improve the above performance with PCA

In [ ]:
# EXPERIMENT B: PCA Analysis - Improving Baseline Performance
print("="*80)
print("EXPERIMENT B: PCA ANALYSIS - IMPROVING BASELINE PERFORMANCE")
print("="*80)

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load final cleaned dataset
df_final = pd.read_csv('data/processed/data_c4_final_recreated_cleaned.csv')
print(f"Dataset shape: {df_final.shape}")

# Use same feature exclusion as Experiment A (no AQ features)
aq_features = [col for col in df_final.columns if 'aq_' in col.lower()]
aq_interaction_features = [col for col in df_final.columns if 'aq' in col.lower() and col not in aq_features]
leakage_features = ['high_aq']
features_to_exclude = ['autism_target'] + aq_features + aq_interaction_features + leakage_features

feature_cols = [col for col in df_final.columns if col not in features_to_exclude]
print(f"Feature columns ({len(feature_cols)}): {feature_cols[:10]}...")

X = df_final[feature_cols]
y = df_final['autism_target']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle data types
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    for col in categorical_cols:
        X[col] = pd.Categorical(X[col]).codes

# Function to apply PCA to questionnaire items
def apply_pca_to_questionnaire(df, items, questionnaire_name, n_components=None):
    if not items:
        print(f"No {questionnaire_name} items found")
        return df, None
    
    print(f"\n{questionnaire_name} items: {len(items)} - {items}")
    
    # Get questionnaire data
    questionnaire_data = df[items]
    
    # Scale the data
    scaler = StandardScaler()
    questionnaire_scaled = scaler.fit_transform(questionnaire_data)
    
    # Determine number of components (keep 80% variance)
    if n_components is None:
        pca_temp = PCA()
        pca_temp.fit(questionnaire_scaled)
        cumsum_variance = np.cumsum(pca_temp.explained_variance_ratio_)
        n_components = np.argmax(cumsum_variance >= 0.8) + 1
    
    # Apply PCA
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(questionnaire_scaled)
    
    # Create new column names
    pca_columns = [f"{questionnaire_name.lower()}_pca_{i+1}" for i in range(n_components)]
    
    # Add PCA results to dataframe
    for i, col in enumerate(pca_columns):
        df[col] = pca_result[:, i]
    
    # Remove original questionnaire items
    df = df.drop(columns=items)
    
    print(f"{questionnaire_name} PCA Results:")
    print(f"Components: {n_components}")
    print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
    print(f"Cumulative variance: {np.cumsum(pca.explained_variance_ratio_)}")
    
    return df, pca_columns

# Apply PCA to each questionnaire
print("\nApplying PCA to questionnaires...")

# SPQ items
spq_items = [col for col in X.columns if col.startswith('spq_') and not col.endswith('_total')]
X_pca, spq_pca_cols = apply_pca_to_questionnaire(X.copy(), spq_items, 'SPQ')

# EQ items  
eq_items = [col for col in X.columns if col.startswith('eq_') and not col.endswith('_total')]
X_pca, eq_pca_cols = apply_pca_to_questionnaire(X_pca, eq_items, 'EQ')

# SQR items
sqr_items = [col for col in X.columns if col.startswith('sqr_') and not col.endswith('_total')]
X_pca, sqr_pca_cols = apply_pca_to_questionnaire(X_pca, sqr_items, 'SQR')

print(f"\nAfter PCA transformation:")
print(f"Original features: {len(feature_cols)}")
print(f"New features: {len(X_pca.columns)}")
print(f"Features dropped: {len(feature_cols) - len(X_pca.columns)}")
print(f"PCA components added: {len(spq_pca_cols) + len(eq_pca_cols) + len(sqr_pca_cols)}")

# Scale all features
scaler = StandardScaler()
X_pca_scaled = scaler.fit_transform(X_pca)
X_pca_scaled = pd.DataFrame(X_pca_scaled, columns=X_pca.columns, index=X_pca.index)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_pca_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nFinal feature matrix: {X_pca_scaled.shape}")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

# Train and evaluate models
print("\n" + "="*60)
print("TRAINING MODELS WITH PCA FEATURES")
print("="*60)

results_pca = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results_pca[name] = {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  AUC: {auc:.4f}")

# Summary of results
print("\n" + "="*60)
print("PCA EXPERIMENT SUMMARY")
print("="*60)

results_pca_df = pd.DataFrame(results_pca).T
results_pca_df = results_pca_df.sort_values('auc', ascending=False)

print("\nPCA Results (ranked by AUC):")
print(results_pca_df.round(4))

# Compare with baseline (Experiment A)
print("\n" + "="*60)
print("COMPARISON WITH BASELINE (EXPERIMENT A)")
print("="*60)

# Load baseline results
baseline_results = pd.read_csv('data/processed/baseline_models_complete_aq_exclusion_results.csv', index_col=0)

comparison_df = pd.DataFrame({
    'Baseline_AUC': baseline_results['auc'],
    'PCA_AUC': results_pca_df['auc'],
    'Baseline_F1': baseline_results['f1'],
    'PCA_F1': results_pca_df['f1']
})
comparison_df['AUC_Improvement'] = comparison_df['PCA_AUC'] - comparison_df['Baseline_AUC']
comparison_df['F1_Improvement'] = comparison_df['PCA_F1'] - comparison_df['Baseline_F1']

print("Performance Comparison:")
print(comparison_df.round(4))

# Find best improvements
best_auc_improvement = comparison_df['AUC_Improvement'].max()
best_f1_improvement = comparison_df['F1_Improvement'].max()

print(f"\nBest AUC improvement: {best_auc_improvement:.4f}")
print(f"Best F1 improvement: {best_f1_improvement:.4f}")

if best_auc_improvement > 0.01:
    print("✅ PCA significantly improved performance!")
elif best_auc_improvement > 0:
    print("✅ PCA slightly improved performance")
else:
    print("❌ PCA did not improve performance")

# Save results
results_pca_df.to_csv('data/processed/pca_experiment_results.csv')
comparison_df.to_csv('data/processed/pca_vs_baseline_comparison.csv')

print(f"\nResults saved to:")
print(f"  - data/processed/pca_experiment_results.csv")
print(f"  - data/processed/pca_vs_baseline_comparison.csv")

print("\n" + "="*80)
print("EXPERIMENT B COMPLETE")
print("="*80)

**PCA didnt do much** likely because larger dataset and better data quality than previous experiments

### C. Threshold Optimization Experiment

In [ ]:
# EXPERIMENT C: Threshold Optimization
print("="*80)
print("EXPERIMENT C: THRESHOLD OPTIMIZATION")
print("="*80)

from sklearn.metrics import precision_recall_curve, roc_curve
from sklearn.model_selection import cross_val_predict
import numpy as np
import matplotlib.pyplot as plt

# Load final cleaned dataset
df_final = pd.read_csv('data/processed/data_c4_final_recreated_cleaned.csv')
print(f"Dataset shape: {df_final.shape}")

# Use same feature exclusion as Experiment A (no AQ features)
aq_features = [col for col in df_final.columns if 'aq_' in col.lower()]
aq_interaction_features = [col for col in df_final.columns if 'aq' in col.lower() and col not in aq_features]
leakage_features = ['high_aq']
features_to_exclude = ['autism_target'] + aq_features + aq_interaction_features + leakage_features

feature_cols = [col for col in df_final.columns if col not in features_to_exclude]
print(f"Feature columns ({len(feature_cols)}): {feature_cols[:10]}...")

X = df_final[feature_cols]
y = df_final['autism_target']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle data types
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    for col in categorical_cols:
        X[col] = pd.Categorical(X[col]).codes

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

# Function to find optimal threshold for a given metric
def find_optimal_threshold(y_true, y_proba, metric='f1'):
    """
    Find optimal threshold for a given metric
    """
    if metric == 'f1':
        precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        optimal_idx = np.argmax(f1_scores)
        optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
        optimal_score = f1_scores[optimal_idx]
        
    elif metric == 'precision':
        precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
        optimal_idx = np.argmax(precision)
        optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
        optimal_score = precision[optimal_idx]
        
    elif metric == 'recall':
        precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
        optimal_idx = np.argmax(recall)
        optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
        optimal_score = recall[optimal_idx]
        
    elif metric == 'balanced_accuracy':
        fpr, tpr, thresholds = roc_curve(y_true, y_proba)
        balanced_acc = (tpr + (1 - fpr)) / 2
        optimal_idx = np.argmax(balanced_acc)
        optimal_threshold = thresholds[optimal_idx]
        optimal_score = balanced_acc[optimal_idx]
        
    else:
        raise ValueError(f"Unknown metric: {metric}")
    
    return optimal_threshold, optimal_score

# Train models and optimize thresholds
print("\n" + "="*60)
print("TRAINING MODELS AND OPTIMIZING THRESHOLDS")
print("="*60)

results = {}
threshold_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Get probabilities on test set
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Default threshold (0.5) results
    y_pred_default = (y_pred_proba >= 0.5).astype(int)
    default_accuracy = accuracy_score(y_test, y_pred_default)
    default_f1 = f1_score(y_test, y_pred_default)
    default_precision = precision_score(y_test, y_pred_default)
    default_recall = recall_score(y_test, y_pred_default)
    default_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Find optimal thresholds for different metrics
    optimal_thresholds = {}
    optimal_scores = {}
    
    for metric in ['f1', 'precision', 'recall', 'balanced_accuracy']:
        threshold, score = find_optimal_threshold(y_test, y_pred_proba, metric)
        optimal_thresholds[metric] = threshold
        optimal_scores[metric] = score
        
        # Calculate performance with optimal threshold
        y_pred_optimal = (y_pred_proba >= threshold).astype(int)
        optimal_accuracy = accuracy_score(y_test, y_pred_optimal)
        optimal_f1 = f1_score(y_test, y_pred_optimal)
        optimal_precision = precision_score(y_test, y_pred_optimal)
        optimal_recall = recall_score(y_test, y_pred_optimal)
        
        print(f"  {metric.capitalize()} optimization:")
        print(f"    Optimal threshold: {threshold:.4f}")
        print(f"    {metric.capitalize()} score: {score:.4f}")
        print(f"    Accuracy: {optimal_accuracy:.4f}")
        print(f"    F1: {optimal_f1:.4f}")
        print(f"    Precision: {optimal_precision:.4f}")
        print(f"    Recall: {optimal_recall:.4f}")
    
    # Store results
    results[name] = {
        'default_accuracy': default_accuracy,
        'default_f1': default_f1,
        'default_precision': default_precision,
        'default_recall': default_recall,
        'default_auc': default_auc,
        'optimal_thresholds': optimal_thresholds,
        'optimal_scores': optimal_scores
    }
    
    threshold_results[name] = {
        'f1_threshold': optimal_thresholds['f1'],
        'precision_threshold': optimal_thresholds['precision'],
        'recall_threshold': optimal_thresholds['recall'],
        'balanced_acc_threshold': optimal_thresholds['balanced_accuracy']
    }

# Summary of threshold optimization results
print("\n" + "="*60)
print("THRESHOLD OPTIMIZATION SUMMARY")
print("="*60)

threshold_df = pd.DataFrame(threshold_results).T
print("\nOptimal Thresholds for Each Model:")
print(threshold_df.round(4))

# Compare default vs optimized performance
print("\n" + "="*60)
print("PERFORMANCE COMPARISON: DEFAULT vs OPTIMIZED")
print("="*60)

comparison_results = []

for name, result in results.items():
    # Get optimized performance using F1-optimized threshold
    f1_threshold = result['optimal_thresholds']['f1']
    model = models[name]
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred_optimized = (y_pred_proba >= f1_threshold).astype(int)
    
    optimized_accuracy = accuracy_score(y_test, y_pred_optimized)
    optimized_f1 = f1_score(y_test, y_pred_optimized)
    optimized_precision = precision_score(y_test, y_pred_optimized)
    optimized_recall = recall_score(y_test, y_pred_optimized)
    
    comparison_results.append({
        'Model': name,
        'Default_F1': result['default_f1'],
        'Optimized_F1': optimized_f1,
        'F1_Improvement': optimized_f1 - result['default_f1'],
        'Default_Precision': result['default_precision'],
        'Optimized_Precision': optimized_precision,
        'Precision_Improvement': optimized_precision - result['default_precision'],
        'Default_Recall': result['default_recall'],
        'Optimized_Recall': optimized_recall,
        'Recall_Improvement': optimized_recall - result['default_recall'],
        'Optimal_Threshold': f1_threshold
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values('F1_Improvement', ascending=False)

print("\nPerformance Improvements (F1-optimized thresholds):")
print(comparison_df.round(4))

# Find best improvements
best_f1_improvement = comparison_df['F1_Improvement'].max()
best_precision_improvement = comparison_df['Precision_Improvement'].max()
best_recall_improvement = comparison_df['Recall_Improvement'].max()

print(f"\nBest Improvements:")
print(f"F1-score: {best_f1_improvement:.4f}")
print(f"Precision: {best_precision_improvement:.4f}")
print(f"Recall: {best_recall_improvement:.4f}")

if best_f1_improvement > 0.01:
    print("✅ Threshold optimization significantly improved F1-score!")
elif best_f1_improvement > 0:
    print("✅ Threshold optimization slightly improved F1-score")
else:
    print("❌ Threshold optimization did not improve F1-score")

# Save results
threshold_df.to_csv('data/processed/threshold_optimization_results.csv')
comparison_df.to_csv('data/processed/threshold_performance_comparison.csv')

print(f"\nResults saved to:")
print(f"  - data/processed/threshold_optimization_results.csv")
print(f"  - data/processed/threshold_performance_comparison.csv")

print("\n" + "="*80)
print("EXPERIMENT C COMPLETE")
print("="*80)

## CORRECTED PIPELINE SUMMARY

### Key Fixes Applied:

1. **Questionnaire Scoring Corrections:**
   - **SPQ-10**: Now correctly scored 0-3 per item (range 0-30)
   - **EQ-10**: Now correctly scored binary 0-1 per item (range 0-10)  
   - **SQR-10**: Now correctly scored binary 0-1 per item (range 0-10)
   - **AQ-10**: Already correct (range 0-10)

2. **Standardization Fix:**
   - Individual questionnaire items are standardized
   - Raw totals (spq_total, eq_total, sqr_total, aq_total) are preserved for filtering
   - This ensures AQ threshold of 6 works correctly

3. **Filtering Logic:**
   - Uses raw AQ scores (not standardized) for threshold comparison
   - Removes autism cases with AQ < 6
   - Preserves both autism and non-autism cases in final dataset

4. **Post-Filtering Rebalancing (CRITICAL FIX):**
   - After filtering, rebalances dataset to 50/50 distribution
   - Matches original notebook size (~20,973 samples)
   - Ensures fair comparison with original experiments

### Expected Results:
- SPQ total: 0-30 range
- EQ total: 0-10 range  
- SQR total: 0-10 range
- AQ total: 0-10 range
- Final dataset: ~20,973 samples with 50/50 autism/non-autism distribution
- Should match original `feature_engineering_2.ipynb` results
